In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

ROOT_DIR        = Path.cwd()
DATA_DIR        = ROOT_DIR / "morse_dataset_public" / "morse_dataset"
TRAIN_AUDIO_DIR = DATA_DIR / "train"
TEST_AUDIO_DIR  = DATA_DIR / "test"
TRAIN_SPEC_DIR  = DATA_DIR / "train_specs"  
TEST_SPEC_DIR   = DATA_DIR / "test_specs"
TRAIN_CSV       = TRAIN_AUDIO_DIR / "labels.csv"
SAMPLE_SUB_CSV  = ROOT_DIR / "sample_submission.csv"
SUBMISSION_CSV  = ROOT_DIR / "submission.csv"
BEST_MODEL_PATH = ROOT_DIR / "best_morse_model.pt"
HISTORY_CSV     = ROOT_DIR / "training_history.csv"

SEED          = 42
BATCH_SIZE    = 64
VALID_SIZE    = 0.05
SAMPLE_RATE   = 8_000
MEL_F_MIN     = 300   
MEL_F_MAX     = 1000  
WAV_HEADER_BYTES = 44
NUM_WORKERS   = 0
PREFETCH_FACTOR = 4
AUGMENT_TRAIN = True

INFERENCE_BLANK_PENALTY = 0.75
INFERENCE_SPACE_PENALTY = 0.0

STAGES = [
    (15, (0.00, 0.05), 1e-3),   
    (15, (0.02, 0.18), 5e-4), 
    (20, (0.10, 0.60), 2e-4), 
    (20, (0.50, 2.00), 1e-4),   
]
PATIENCE   = 8
MIN_DELTA  = 0.0

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

for path in [TRAIN_CSV, SAMPLE_SUB_CSV, TRAIN_AUDIO_DIR, TEST_AUDIO_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Не найдено: {path}")

train_labels      = pd.read_csv(TRAIN_CSV,       dtype={"filename": "string", "text": "string"})
sample_submission = pd.read_csv(SAMPLE_SUB_CSV,  dtype={"filename": "string", "text": "string"})

print(f"Device:    {DEVICE}")
print(f"AMP:       {USE_AMP}")
print(f"Train rows: {len(train_labels):,}")
print(f"Test rows:  {len(sample_submission):,}")
display(train_labels.head())

Device:    cuda
AMP:       True
Train rows: 30,000
Test rows:  5,000


,filename,text
0,00000.wav,7 8 1 0
1,00001.wav,379466
2,00002.wav,8 7 0 2
3,00003.wav,88 19
4,00004.wav,34 176


In [ ]:
VOCAB        = list("0123456789 ")
CHAR_MAP     = {"<blank>": 0}
CHAR_MAP.update({char: idx for idx, char in enumerate(VOCAB, start=1)})
REV_CHAR_MAP = {idx: char for char, idx in CHAR_MAP.items()}
NUM_CLASSES  = len(CHAR_MAP)


def encode_text(text: str) -> torch.Tensor:
    text = str(text)
    unknown = sorted(set(text) - set(VOCAB))
    if unknown:
        raise ValueError(f"Неизвестные символы в {text!r}: {unknown}")
    return torch.tensor([CHAR_MAP[c] for c in text], dtype=torch.long)


def decode_token_ids(token_ids) -> str:
    return "".join(REV_CHAR_MAP[int(t)] for t in token_ids)


def decode_targets(flat_targets, target_lengths) -> list:
    decoded, offset = [], 0
    for length in target_lengths.tolist():
        decoded.append(decode_token_ids(flat_targets[offset: offset + length].tolist()))
        offset += length
    return decoded


def levenshtein_distance(pred: str, target: str) -> int:
    if pred == target:
        return 0
    if len(pred) < len(target):
        pred, target = target, pred
    prev = list(range(len(target) + 1))
    for i, pc in enumerate(pred, 1):
        curr = [i]
        for j, tc in enumerate(target, 1):
            curr.append(min(curr[j-1]+1, prev[j]+1, prev[j-1]+int(pc != tc)))
        prev = curr
    return prev[-1]


def mean_levenshtein(preds, targets):
    dists = [levenshtein_distance(p, t) for p, t in zip(preds, targets)]
    norm  = [d / max(1, len(t)) for d, t in zip(dists, targets)]
    return float(np.mean(dists)), float(np.mean(norm))


print(f"Classes: {NUM_CLASSES} -> {CHAR_MAP}")

Classes: 12 -> {'<blank>': 0, '0': 1, '1': 2, '2': 3, '3': 4, '4': 5, '5': 6, '6': 7, '7': 8, '8': 9, '9': 10, ' ': 11}


## 4. Аугментации, датасет и DataLoader
**Ключевое решение:** шум применяется на уровне waveform ДО mel-transform (физически корректно).
Нормализация спектрограммы делается ПОСЛЕ добавления шума — модель видит реалистичный сигнал.

In [ ]:
MEL_CONFIG = dict(
    sample_rate=SAMPLE_RATE,
    n_fft=1024,     
    hop_length=80,
    n_mels=32,
    f_min=MEL_F_MIN,
    f_max=MEL_F_MAX,
    power=2.0,
)


def load_wav(audio_path) -> torch.Tensor:
    audio = np.fromfile(str(audio_path), dtype="<i2", offset=WAV_HEADER_BYTES).astype(np.float32)
    return torch.from_numpy(audio / 32768.0).unsqueeze(0) 



def awgn(waveform: torch.Tensor, noise_rms: float) -> torch.Tensor:
    return waveform + torch.randn_like(waveform) * noise_rms


def impulsive_noise(waveform: torch.Tensor, noise_rms: float) -> torch.Tensor:
    exponent = random.uniform(0.3, 0.6)
    base     = torch.randn_like(waveform)
    noise    = torch.sign(base) * base.abs().pow(exponent)
    noise    = noise / (noise.abs().max() + 1e-8)
    return waveform + noise * noise_rms


def fading(waveform: torch.Tensor) -> torch.Tensor:
    T        = waveform.shape[-1]
    n_points = random.randint(4, 16)      
    knots    = torch.rand(1, 1, n_points)   
    envelope = torch.nn.functional.interpolate(
        knots, size=T, mode="linear", align_corners=False
    ).squeeze()
    envelope = envelope.clamp(0.05, 1.0)
    return waveform * envelope


def colored_noise(length: int, alpha: float = 1.0) -> torch.Tensor:
    freqs    = torch.fft.rfftfreq(length).clamp(min=1e-6)
    power    = freqs ** (-alpha / 2.0)
    phases   = torch.rand(len(freqs)) * 2 * 3.14159265
    spectrum = power * torch.exp(1j * phases)
    noise    = torch.fft.irfft(spectrum, n=length)
    noise    = noise / (noise.abs().max() + 1e-8)
    return noise.unsqueeze(0)


def apply_radio_augmentation(waveform: torch.Tensor, noise_range: tuple) -> torch.Tensor:
    waveform   = waveform * random.uniform(0.75, 1.35)
    signal_rms = waveform.square().mean().sqrt().clamp(min=1e-4)
    noise_rms  = float(signal_rms) * random.uniform(*noise_range)

    fading_prob = 0.3 + 0.4 * (noise_range[1] > 0.3)
    if random.random() < fading_prob:
        waveform = fading(waveform)

    r = random.random()
    if r < 0.40:
        waveform = awgn(waveform, noise_rms)
    elif r < 0.70:
        waveform = impulsive_noise(waveform, noise_rms)
    else:
        alpha    = random.uniform(0.5, 2.0)
        noise    = colored_noise(waveform.shape[-1], alpha=alpha)
        waveform = waveform + noise * noise_rms

    return waveform.clamp(-1.0, 1.0)


def random_pad(waveform: torch.Tensor, noise_range: tuple = (0.0, 0.05)) -> torch.Tensor:
    max_len     = int(8.0 * SAMPLE_RATE)
    current_len = waveform.shape[-1]
    if current_len >= max_len:
        return waveform
    target_len = random.randint(current_len, max_len)
    pad_total  = target_len - current_len
    pad_left   = random.randint(0, pad_total)
    pad_right  = pad_total - pad_left
    noise_level = random.uniform(*noise_range) if noise_range[1] > 0 else 0.02
    pad_noise   = torch.randn(1, pad_total) * noise_level
    padded      = torch.cat([
        pad_noise[:, :pad_left],
        waveform,
        pad_noise[:, pad_left:],
    ], dim=1)
    return padded


class MorseDataset(Dataset):
    def __init__(self, df, audio_dir, has_targets=True, augment=False, noise_range=(0.0, 0.05)):
        self.df          = df.reset_index(drop=True)
        self.audio_dir   = Path(audio_dir)
        self.has_targets = has_targets
        self.augment     = augment
        self.noise_range = noise_range

        self.mel_transform   = T.MelSpectrogram(**MEL_CONFIG)
        self.amplitude_to_db = T.AmplitudeToDB()
        self.freq_mask       = T.FrequencyMasking(freq_mask_param=4)
        self.time_mask       = T.TimeMasking(time_mask_param=20)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        waveform = load_wav(self.audio_dir / str(row["filename"]))

        if self.augment:
            if random.random() < 0.60:
                waveform = random_pad(waveform, self.noise_range)
            if random.random() < 0.80:
                waveform = apply_radio_augmentation(waveform, self.noise_range)

        mel = self.mel_transform(waveform)
        mel = self.amplitude_to_db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        if self.augment:
            if random.random() < 0.35:
                mel = self.freq_mask(mel)
            if random.random() < 0.35:
                mel = self.time_mask(mel)

        spec = mel.squeeze(0)  

        target = (
            encode_text(row["text"])
            if self.has_targets and "text" in row and pd.notna(row["text"])
            else torch.empty(0, dtype=torch.long)
        )
        return spec, target


def collate_fn(batch):
    specs, targets = zip(*batch)

    specs_t      = [s.permute(1, 0) for s in specs]
    input_lengths = torch.tensor([s.shape[0] for s in specs_t], dtype=torch.long)
    padded_specs = nn.utils.rnn.pad_sequence(specs_t, batch_first=True, padding_value=0.0)
    padded_specs = padded_specs.permute(0, 2, 1).unsqueeze(1).contiguous()

    target_lengths = torch.tensor([t.numel() for t in targets], dtype=torch.long)
    flat_targets   = (
        torch.cat([t for t in targets if t.numel() > 0])
        if target_lengths.sum() > 0 else torch.empty(0, dtype=torch.long)
    )
    return padded_specs, flat_targets, input_lengths, target_lengths


def make_loader(dataset, batch_size, shuffle):
    kwargs = dict(
        batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collate_fn,
    )
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"]    = PREFETCH_FACTOR
    return DataLoader(dataset, **kwargs)


print("Dataset / DataLoader готовы.")
print("Шумы: AWGN (тепловой) | Импульсный (молнии) | Цветной (фон) | Фэйдинг (замирание)")


Dataset / DataLoader готовы.
Шумы: AWGN (тепловой) | Импульсный (молнии) | Цветной (фон) | Фэйдинг (замирание)


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 1), dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=pool),
            nn.Dropout2d(dropout),
        )

    def forward(self, x):
        return self.block(x)


class CRNNMorse(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, rnn_hidden=128, dropout=0.18):
        super().__init__()
        self.cnn = nn.Sequential(
            ConvBlock(1,  32, pool=(2, 2), dropout=0.05),
            ConvBlock(32, 64, pool=(2, 1), dropout=0.08),
            ConvBlock(64, 96, pool=(2, 1), dropout=0.10),
        )
        self.bridge = nn.Sequential(
            nn.Linear(96 * 4, rnn_hidden),
            nn.LayerNorm(rnn_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.rnn = nn.GRU(
            input_size=rnn_hidden, hidden_size=rnn_hidden,
            num_layers=2, bidirectional=True,
            batch_first=True, dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(rnn_hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(rnn_hidden * 2, num_classes),
        )

    def forward(self, x):
        x = self.cnn(x)
        B, C, F, T = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(B, T, C * F)
        x = self.bridge(x)
        x, _ = self.rnn(x)
        x = self.classifier(x)
        return x.permute(1, 0, 2) 

def cnn_output_lengths(input_lengths: torch.Tensor) -> torch.Tensor:
    return torch.div(input_lengths, 2, rounding_mode="floor").clamp(min=1)


def greedy_decoder(
    logits, rev_char_map, input_lengths=None,
    blank_penalty=0.0, space_penalty=0.0
) -> list:

    if blank_penalty or space_penalty:
        logits = logits.clone()
        logits[:, :, 0] -= blank_penalty
        logits[:, :, CHAR_MAP[" "]] -= space_penalty

    token_ids = torch.argmax(logits, dim=2).transpose(0, 1)
    out_lens  = (
        cnn_output_lengths(input_lengths.cpu()).clamp(max=token_ids.size(1))
        if input_lengths is not None
        else torch.full((token_ids.size(0),), token_ids.size(1), dtype=torch.long)
    )

    decoded = []
    for seq, rlen in zip(token_ids, out_lens.tolist()):
        chars, prev = [], None
        for tok in seq[:rlen].tolist():
            if tok != 0 and tok != prev:
                chars.append(rev_char_map[tok])
            prev = tok
        decoded.append("".join(chars))
    return decoded


model = CRNNMorse(num_classes=NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Параметры модели: {n_params:,}")
print(model)

In [ ]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
scaler    = GradScaler(enabled=USE_AMP)


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=MIN_DELTA, path=BEST_MODEL_PATH):
        self.patience   = patience
        self.min_delta  = min_delta
        self.path       = Path(path)
        self.best_score = float("inf")
        self.counter    = 0

    def step(self, score, model) -> bool:
        if score < self.best_score - self.min_delta:
            self.best_score = score
            self.counter    = 0
            torch.save(model.state_dict(), self.path)
            return False
        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for specs, targets, in_lens, tg_lens in loader:
        specs   = specs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits      = model(specs)
            log_probs   = logits.log_softmax(dim=2)
            out_lens    = cnn_output_lengths(in_lens).clamp(max=log_probs.size(0))
            loss        = criterion(log_probs, targets, out_lens, tg_lens)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))


def validate(model, loader):
    model.eval()
    total_loss, all_preds, all_tgts = 0.0, [], []
    with torch.no_grad():
        for specs, targets, in_lens, tg_lens in loader:
            specs        = specs.to(DEVICE, non_blocking=True)
            tgts_device  = targets.to(DEVICE, non_blocking=True)

            with autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits    = model(specs)
                log_probs = logits.log_softmax(dim=2)
                out_lens  = cnn_output_lengths(in_lens).clamp(max=log_probs.size(0))
                loss      = criterion(log_probs, tgts_device, out_lens, tg_lens)
            total_loss += loss.item()

            all_preds.extend(p.strip() for p in greedy_decoder(log_probs.detach().cpu(), REV_CHAR_MAP, in_lens))
            all_tgts.extend(decode_targets(targets, tg_lens))

    lev, cer = mean_levenshtein(all_preds, all_tgts)
    return {
        "loss": total_loss / max(1, len(loader)),
        "lev":  lev,
        "cer":  cer,
        "examples": list(zip(all_preds, all_tgts)),
        "mistakes": [(p, t) for p, t in zip(all_preds, all_tgts) if p != t][:8],
    }


def show_examples(metrics, n=5):
    print("  Примеры:")
    for pred, target in metrics["examples"][:n]:
        marker = "✓" if pred == target else "✗"
        print(f"    {marker} pred={pred!r:30s} | target={target!r}")
    if metrics["mistakes"]:
        print("  Ошибки:")
        for pred, target in metrics["mistakes"][:3]:
            print(f"    ✗ pred={pred!r:30s} | target={target!r}")


print("Функции обучения готовы.")

## 7. Запуск стадийного обучения

In [ ]:
train_df, val_df = train_test_split(
    train_labels, test_size=VALID_SIZE, random_state=SEED, shuffle=True
)

history       = []
epoch_global  = 0
early_stop    = EarlyStopping()

for stage_idx, (n_epochs, noise_range, lr) in enumerate(STAGES, 1):
    print(f"\n{'='*65}")
    print(f"  Стадия {stage_idx}: эпох={n_epochs}, шум={noise_range}, lr={lr}")
    print(f"{'='*65}")

    if stage_idx > 1 and BEST_MODEL_PATH.exists():
        print(f"  → Загружаем лучшие веса перед стадией {stage_idx}...")
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
        early_stop = EarlyStopping() 

    train_dataset     = MorseDataset(train_df, TRAIN_AUDIO_DIR, augment=True,  noise_range=noise_range)
    val_dataset       = MorseDataset(val_df,   TRAIN_AUDIO_DIR, augment=False, noise_range=(0,0))
    hard_val_dataset  = MorseDataset(val_df,   TRAIN_AUDIO_DIR, augment=True,  noise_range=noise_range)

    train_loader    = make_loader(train_dataset,    BATCH_SIZE, shuffle=True)
    val_loader      = make_loader(val_dataset,      BATCH_SIZE, shuffle=False)
    hard_val_loader = make_loader(hard_val_dataset, BATCH_SIZE, shuffle=False)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    for epoch in range(1, n_epochs + 1):
        epoch_global += 1

        tr_loss  = train_one_epoch(model, train_loader, optimizer)
        val_m    = validate(model, val_loader)
        hard_m   = validate(model, hard_val_loader)
        scheduler.step(hard_m["lev"])

        stopped = early_stop.step(hard_m["lev"], model)
        cur_lr  = optimizer.param_groups[0]["lr"]

        row = dict(
            epoch=epoch_global, stage=stage_idx,
            train_loss=tr_loss,
            val_loss=val_m["loss"],   val_lev=val_m["lev"],   val_cer=val_m["cer"],
            hard_loss=hard_m["loss"], hard_lev=hard_m["lev"], hard_cer=hard_m["cer"],
            lr=cur_lr, patience_counter=early_stop.counter,
        )
        history.append(row)

        print(
            f"[S{stage_idx}] Ep {epoch:02d}/{n_epochs} | "
            f"tr_loss={tr_loss:.4f} | "
            f"val_lev={val_m['lev']:.4f} val_cer={val_m['cer']:.4f} | "
            f"hard_lev={hard_m['lev']:.4f} hard_cer={hard_m['cer']:.4f} | "
            f"lr={cur_lr:.1e} | patience={early_stop.counter}/{early_stop.patience}"
        )

        if epoch == 1 or epoch % 5 == 0 or stopped:
            show_examples(val_m)

        if stopped:
            print(f"  ⏹ EarlyStopping: best hard_lev={early_stop.best_score:.4f}")
            break

if BEST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    print(f"\n Лучшая модель загружена из {BEST_MODEL_PATH}")

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_CSV, index=False)
print(f" История сохранена: {HISTORY_CSV}")
display(history_df.tail(10))

In [ ]:
if BEST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    print(f"Веса загружены из {BEST_MODEL_PATH}")

model.eval()

test_df      = sample_submission.copy()
test_dataset = MorseDataset(test_df, TEST_AUDIO_DIR, has_targets=False, augment=False)
test_loader  = make_loader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []
with torch.no_grad():
    for specs, _, in_lens, _ in test_loader:
        specs = specs.to(DEVICE, non_blocking=True)
        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits    = model(specs)
            log_probs = logits.log_softmax(dim=2)

        all_predictions.extend(
            p.strip() for p in greedy_decoder(
                log_probs.detach().cpu(), REV_CHAR_MAP, in_lens,
                blank_penalty=INFERENCE_BLANK_PENALTY,
                space_penalty=INFERENCE_SPACE_PENALTY,
            )
        )

raw = pd.Series(all_predictions, dtype="string")
test_df["text"] = raw.str.strip()

empty = int((test_df["text"].fillna("") == "").sum())
if empty:
    test_df.loc[test_df["text"].fillna("") == "", "text"] = "0"

test_df.to_csv(SUBMISSION_CSV, index=False)

print(f"Submission сохранён: {SUBMISSION_CSV}")
print(f"Пустых предсказаний: {empty}")
print(f"Средняя длина предсказания: {test_df['text'].str.len().mean():.2f}")
print(test_df["text"].value_counts(dropna=False).head(15).to_string())
display(test_df.head(10))